In [22]:
import pandas as pd
import numpy as np
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

def export_comprehensive_data_insights(file_path, output_file="Data_Insights_Report.xlsx"):
    """
    Analyze Excel file and export comprehensive insights to Excel with multiple sheets
    """
    
    print("Loading and analyzing your Excel file...")
    
    # Load Excel file
    excel_file = pd.ExcelFile(file_path)
    sheet_name = excel_file.sheet_names[0]  # Use first sheet
    df = pd.read_excel(file_path, sheet_name=sheet_name)
    
    print(f"Loaded sheet: '{sheet_name}' with {df.shape[0]} rows and {df.shape[1]} columns")
    
    # Create Excel writer object
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        
        # 1. BASIC INFORMATION SHEET
        print("Generating basic information sheet...")
        basic_info = {
            'Metric': ['File Name', 'Sheet Name', 'Total Rows', 'Total Columns', 'Total Cells', 'Memory Usage (MB)'],
            'Value': [
                file_path,
                sheet_name,
                df.shape[0],
                df.shape[1],
                df.shape[0] * df.shape[1],
                round(df.memory_usage(deep=True).sum() / 1024 / 1024, 2)
            ]
        }
        pd.DataFrame(basic_info).to_excel(writer, sheet_name='Basic_Info', index=False)
        
        # 2. DETAILED COLUMN ANALYSIS SHEET
        print("Generating detailed column analysis...")
        columns_analysis = []
        
        for col in df.columns:
            col_info = {
                'Column_Name': col,
                'Data_Type': str(df[col].dtype),
                'Non_Null_Count': df[col].count(),
                'Null_Count': df[col].isnull().sum(),
                'Null_Percentage': round(df[col].isnull().sum() / len(df) * 100, 2),
                'Unique_Values': df[col].nunique(),
                'Uniqueness_Percentage': round(df[col].nunique() / len(df) * 100, 2)
            }
            
            # Add numerical statistics if applicable
            if pd.api.types.is_numeric_dtype(df[col]):
                col_info.update({
                    'Min_Value': df[col].min(),
                    'Max_Value': df[col].max(),
                    'Mean': round(df[col].mean(), 2),
                    'Median': round(df[col].median(), 2),
                    'Std_Deviation': round(df[col].std(), 2),
                    'Zero_Count': (df[col] == 0).sum(),
                    'Negative_Count': (df[col] < 0).sum()
                })
                
                # Calculate potential outliers
                Q1 = df[col].quantile(0.25)
                Q3 = df[col].quantile(0.75)
                IQR = Q3 - Q1
                outliers = ((df[col] < (Q1 - 1.5 * IQR)) | (df[col] > (Q3 + 1.5 * IQR))).sum()
                col_info['Potential_Outliers'] = outliers
            else:
                # For text columns, add text-specific metrics
                non_null_text = df[col].dropna().astype(str)
                if len(non_null_text) > 0:
                    col_info.update({
                        'Avg_Text_Length': round(non_null_text.str.len().mean(), 1),
                        'Max_Text_Length': non_null_text.str.len().max(),
                        'Min_Text_Length': non_null_text.str.len().min(),
                        'Leading_Spaces': non_null_text.str.startswith(' ').sum(),
                        'Trailing_Spaces': non_null_text.str.endswith(' ').sum(),
                        'All_Uppercase': non_null_text.str.isupper().sum(),
                        'All_Lowercase': non_null_text.str.islower().sum()
                    })
            
            columns_analysis.append(col_info)
        
        df_columns = pd.DataFrame(columns_analysis)
        df_columns.to_excel(writer, sheet_name='Column_Analysis', index=False)
        
        # 3. TOP VALUES FOR EACH COLUMN SHEET
        print("Generating top values analysis...")
        top_values_data = []
        
        for col in df.columns:
            if df[col].dtype == 'object':
                top_vals = df[col].value_counts().head(10)
                for i, (value, count) in enumerate(top_vals.items(), 1):
                    top_values_data.append({
                        'Column': col,
                        'Rank': i,
                        'Value': str(value),
                        'Count': count,
                        'Percentage': round(count / len(df) * 100, 2)
                    })
        
        if top_values_data:
            pd.DataFrame(top_values_data).to_excel(writer, sheet_name='Top_Values', index=False)
        
        # 4. MISSING VALUES ANALYSIS SHEET
        print("Generating missing values analysis...")
        missing_analysis = []
        missing_data = df.isnull().sum()
        
        for col, count in missing_data[missing_data > 0].items():
            missing_analysis.append({
                'Column': col,
                'Missing_Count': count,
                'Missing_Percentage': round(count / len(df) * 100, 2),
                'Data_Type': str(df[col].dtype)
            })
        
        if missing_analysis:
            pd.DataFrame(missing_analysis).to_excel(writer, sheet_name='Missing_Values', index=False)
        else:
            pd.DataFrame([{'Message': 'No missing values found in the dataset!'}]).to_excel(writer, sheet_name='Missing_Values', index=False)
        
        # 5. DUPLICATE ANALYSIS SHEET
        print("Generating duplicate analysis...")
        duplicate_info = []
        
        # Complete duplicate rows
        complete_duplicates = df.duplicated().sum()
        duplicate_info.append({
            'Type': 'Complete Duplicate Rows',
            'Count': complete_duplicates,
            'Percentage': round(complete_duplicates / len(df) * 100, 2)
        })
        
        # Check potential ID columns for duplicates
        potential_id_cols = [col for col in df.columns if 'id' in col.lower() or 'code' in col.lower()]
        for col in potential_id_cols:
            col_duplicates = df[col].duplicated().sum()
            duplicate_info.append({
                'Type': f'Duplicates in {col}',
                'Count': col_duplicates,
                'Percentage': round(col_duplicates / len(df) * 100, 2)
            })
        
        pd.DataFrame(duplicate_info).to_excel(writer, sheet_name='Duplicates', index=False)
        
        # 6. CORRELATION MATRIX SHEET (for numerical columns)
        numerical_cols = df.select_dtypes(include=[np.number]).columns
        if len(numerical_cols) > 1:
            print("Generating correlation matrix...")
            corr_matrix = df[numerical_cols].corr()
            corr_matrix.to_excel(writer, sheet_name='Correlation_Matrix')
            
            # Strong correlations summary
            strong_corr = []
            for i in range(len(corr_matrix.columns)):
                for j in range(i+1, len(corr_matrix.columns)):
                    corr_val = corr_matrix.iloc[i, j]
                    if abs(corr_val) > 0.7:
                        strong_corr.append({
                            'Column_1': corr_matrix.columns[i],
                            'Column_2': corr_matrix.columns[j],
                            'Correlation': round(corr_val, 3),
                            'Strength': 'Strong Positive' if corr_val > 0.7 else 'Strong Negative'
                        })
            
            if strong_corr:
                pd.DataFrame(strong_corr).to_excel(writer, sheet_name='Strong_Correlations', index=False)
        
        # 7. DATA QUALITY SUMMARY SHEET
        print("Generating data quality summary...")
        quality_issues = []
        
        # Missing values
        total_missing = df.isnull().sum().sum()
        if total_missing > 0:
            quality_issues.append({
                'Issue_Type': 'Missing Values',
                'Count': total_missing,
                'Severity': 'High' if total_missing > len(df) * 0.1 else 'Medium',
                'Description': f'{total_missing} missing values across {(df.isnull().sum() > 0).sum()} columns'
            })
        
        # Duplicates
        if complete_duplicates > 0:
            quality_issues.append({
                'Issue_Type': 'Duplicate Rows',
                'Count': complete_duplicates,
                'Severity': 'Medium',
                'Description': f'{complete_duplicates} complete duplicate rows'
            })
        
        # High cardinality categorical columns
        for col in df.select_dtypes(include=['object']).columns:
            unique_ratio = df[col].nunique() / len(df)
            if unique_ratio > 0.5:
                quality_issues.append({
                    'Issue_Type': 'High Cardinality',
                    'Count': df[col].nunique(),
                    'Severity': 'Low',
                    'Description': f'Column {col} has {df[col].nunique()} unique values ({unique_ratio:.1%} uniqueness)'
                })
        
        # Constant columns
        constant_cols = [col for col in df.columns if df[col].nunique() <= 1]
        for col in constant_cols:
            quality_issues.append({
                'Issue_Type': 'Constant Column',
                'Count': 1,
                'Severity': 'Medium',
                'Description': f'Column {col} has only 1 unique value'
            })
        
        if quality_issues:
            pd.DataFrame(quality_issues).to_excel(writer, sheet_name='Quality_Issues', index=False)
        else:
            pd.DataFrame([{'Message': 'No major data quality issues detected!'}]).to_excel(writer, sheet_name='Quality_Issues', index=False)
        
        # 8. CLUSTERING READINESS ASSESSMENT
        print("Generating clustering readiness assessment...")
        clustering_assessment = []
        
        clustering_assessment.append({
            'Metric': 'Total Features',
            'Value': len(df.columns),
            'Status': 'Good' if len(df.columns) >= 3 else 'Needs More Features'
        })
        
        clustering_assessment.append({
            'Metric': 'Numerical Features',
            'Value': len(numerical_cols),
            'Status': 'Good' if len(numerical_cols) >= 2 else 'Limited'
        })
        
        categorical_cols = df.select_dtypes(include=['object']).columns
        clustering_assessment.append({
            'Metric': 'Categorical Features',
            'Value': len(categorical_cols),
            'Status': 'Needs Encoding' if len(categorical_cols) > 0 else 'None'
        })
        
        clustering_assessment.append({
            'Metric': 'Missing Values',
            'Value': f"{total_missing} ({total_missing/df.size:.1%})",
            'Status': 'Needs Attention' if total_missing > 0 else 'Clean'
        })
        
        clustering_assessment.append({
            'Metric': 'Duplicate Rows',
            'Value': complete_duplicates,
            'Status': 'Needs Cleaning' if complete_duplicates > 0 else 'Clean'
        })
        
        pd.DataFrame(clustering_assessment).to_excel(writer, sheet_name='Clustering_Readiness', index=False)
        
        # 9. SAMPLE DATA SHEET
        print("Adding sample data...")
        # First 10 rows
        df.head(10).to_excel(writer, sheet_name='Sample_Data', index=False)
    
    print(f"\n✅ SUCCESS! Comprehensive data insights exported to: {output_file}")
    print(f"\nThe Excel file contains {8 if len(numerical_cols) > 1 else 7} sheets:")
    print("   • Basic_Info - Dataset overview")
    print("   • Column_Analysis - Detailed column statistics")
    print("   • Top_Values - Most frequent values in categorical columns")
    print("   • Missing_Values - Missing data analysis")
    print("   • Duplicates - Duplicate row analysis")
    if len(numerical_cols) > 1:
        print("   • Correlation_Matrix - Numerical correlations")
        print("   • Strong_Correlations - High correlations summary")
    print("   • Quality_Issues - Data quality problems")
    print("   • Clustering_Readiness - Assessment for clustering")
    print("   • Sample_Data - First 10 rows of your data")
    
    return output_file

# Usage: Run this with your Excel file
if __name__ == "__main__":
    # Replace with your Excel file path
    excel_file_path = "GISMS July'25.xlsx"
    output_report = "GISMS_Data_Insights_Report.xlsx"
    
    try:
        export_comprehensive_data_insights(excel_file_path, output_report)
        print(f"\n🎉 Your comprehensive data report is ready: {output_report}")
    except Exception as e:
        print(f"Error: {e}")
        print("Please make sure your Excel file path is correct!")


Loading and analyzing your Excel file...
Loaded sheet: 'Salesforce Overall Registration' with 33785 rows and 30 columns
Error: [Errno 13] Permission denied: 'GISMS_Data_Insights_Report.xlsx'
Please make sure your Excel file path is correct!


In [53]:
df= pd.read_excel("GISMS_Separated1.xlsx")

In [ ]:
def move_others_to_separate_sheet(df, output_file="GISMS_Separated.xlsx"):
    """Move 'Others' rows to a separate sheet in Excel"""
    
    # Separate the data
    others_rows = df[df['Job_Function'] == 'Others'].copy()
    main_data = df[df['Job_Function'] != 'Others'].copy()
    
    # Save to Excel with multiple sheets
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        main_data.to_excel(writer, sheet_name='Main_Data', index=False)
        others_rows.to_excel(writer, sheet_name='Others_Data', index=False)
    
    print(f"Main data: {len(main_data)} rows saved to 'Main_Data' sheet")
    print(f"Others data: {len(others_rows)} rows saved to 'Others_Data' sheet")
    
    return main_data, others_rows
move_others_to_separate_sheet(df)

Main data: 25791 rows saved to 'Main_Data' sheet
Others data: 7994 rows saved to 'Others_Data' sheet


(             Unique ID                         Type  \
 0      KIMS_CEP_191329  Registered but not Attended   
 1      KIMS_CEP_191330  Registered but not Attended   
 4      KIMS_CEP_191333        Registered & Attended   
 9      KIMS_CEP_191338  Registered but not Attended   
 10     KIMS_CEP_191339  Registered but not Attended   
 ...                ...                          ...   
 33778  KIMS_CEP_225107        Registered & Attended   
 33779  KIMS_CEP_225108        Registered & Attended   
 33781  KIMS_CEP_225110        Registered & Attended   
 33782  KIMS_CEP_225111        Registered & Attended   
 33783  KIMS_CEP_225112        Registered & Attended   
 
                                             Program Name Type of Event  \
 0      Salesforce The Great India Sales & Marketing &...       Virtual   
 1      Salesforce The Great India Sales & Marketing &...       Virtual   
 4      Salesforce The Great India Sales & Marketing &...       Virtual   
 9      Salesforce The Gre

In [ ]:
import pandas as pd
import numpy as np
import re
# For  column validation and separation shows what invalid data is there with excel file
def create_comprehensive_invalid_terms_list():
    """
    The most complete list of invalid/placeholder terms based on extensive research
    Includes all categories from data cleaning literature and real-world datasets
    """
    
    invalid_terms = [
        # PREVIOUSLY COVERED TERMS (keeping all the ones from before)...
        # Direct invalid terms - all cases
        'others', 'other', 'unknown', 'not available', 'n/a', 'na', 'null', 
        'none', 'blank', 'empty', 'missing', 'not specified', 'not applicable',
        'miscellaneous', 'misc', 'general', 'various', 'mixed', 'assorted',
        'Others', 'Other', 'Unknown', 'Not Available', 'N/A', 'Na', 'Null', 
        'None', 'Blank', 'Empty', 'Missing', 'Not Specified', 'Not Applicable',
        'Miscellaneous', 'Misc', 'General', 'Various', 'Mixed', 'Assorted',
        'OTHERS', 'OTHER', 'UNKNOWN', 'NOT AVAILABLE', 'N/A', 'NA', 'NULL', 
        'NONE', 'BLANK', 'EMPTY', 'MISSING', 'NOT SPECIFIED', 'NOT APPLICABLE',
        'MISCELLANEOUS', 'MISC', 'GENERAL', 'VARIOUS', 'MIXED', 'ASSORTED',
        
        # NEWLY DISCOVERED CATEGORIES:
        
        # 1. FORM/UI PLACEHOLDERS[2][3] (commonly seen in web forms)
        'please select', 'Please Select', 'PLEASE SELECT',
        'choose option', 'Choose Option', 'CHOOSE OPTION',
        'select one', 'Select One', 'SELECT ONE',
        'pick one', 'Pick One', 'PICK ONE',
        'make selection', 'Make Selection', 'MAKE SELECTION',
        'choose from list', 'Choose From List', 'CHOOSE FROM LIST',
        'select from dropdown', 'Select From Dropdown', 'SELECT FROM DROPDOWN',
        'click to select', 'Click To Select', 'CLICK TO SELECT',
        'no selection', 'No Selection', 'NO SELECTION',
        'no option selected', 'No Option Selected', 'NO OPTION SELECTED',
        
        # 2. TESTING/DEVELOPMENT PLACEHOLDERS[7] (from QA and development)
        'test', 'Test', 'TEST', 'testing', 'Testing', 'TESTING',
        'test123', 'Test123', 'TEST123', 'test data', 'Test Data', 'TEST DATA',
        'abc', 'ABC', 'xyz', 'XYZ', 'abc123', 'ABC123', 'xyz123', 'XYZ123',
        'sample', 'Sample', 'SAMPLE', 'example', 'Example', 'EXAMPLE',
        'demo', 'Demo', 'DEMO', 'prototype', 'Prototype', 'PROTOTYPE',
        'lorem ipsum', 'Lorem Ipsum', 'LOREM IPSUM',
        'dummy data', 'Dummy Data', 'DUMMY DATA',
        'mock', 'Mock', 'MOCK', 'fake data', 'Fake Data', 'FAKE DATA',
        'development', 'Development', 'DEVELOPMENT',
        'dev', 'Dev', 'DEV', 'qa', 'QA', 'quality assurance',
        
        # 3. SURVEY/RESEARCH SPECIFIC[12][14] (from academic and market research)
        'no response', 'No Response', 'NO RESPONSE',
        'declined to answer', 'Declined To Answer', 'DECLINED TO ANSWER',
        'refused', 'Refused', 'REFUSED', 'skipped', 'Skipped', 'SKIPPED',
        'no answer', 'No Answer', 'NO ANSWER',
        'prefer not to say', 'Prefer Not To Say', 'PREFER NOT TO SAY',
        'rather not say', 'Rather Not Say', 'RATHER NOT SAY',
        'no opinion', 'No Opinion', 'NO OPINION',
        'neutral', 'Neutral', 'NEUTRAL', 'indifferent', 'Indifferent', 'INDIFFERENT',
        'dont care', "don't care", 'Dont Care', "Don't Care", 'DONT CARE', "DON'T CARE",
        'no preference', 'No Preference', 'NO PREFERENCE',
        'not applicable to me', 'Not Applicable To Me', 'NOT APPLICABLE TO ME',
        'does not apply', 'Does Not Apply', 'DOES NOT APPLY',
        
        # 4. BUSINESS/CORPORATE SPECIFIC[6] (from business datasets)
        'confidential', 'Confidential', 'CONFIDENTIAL',
        'proprietary', 'Proprietary', 'PROPRIETARY',
        'internal only', 'Internal Only', 'INTERNAL ONLY',
        'classified', 'Classified', 'CLASSIFIED',
        'restricted access', 'Restricted Access', 'RESTRICTED ACCESS',
        'privileged', 'Privileged', 'PRIVILEGED',
        'redacted', 'Redacted', 'REDACTED',
        'withheld', 'Withheld', 'WITHHELD',
        'suppressed', 'Suppressed', 'SUPPRESSED',
        'protected', 'Protected', 'PROTECTED',
        'sensitive', 'Sensitive', 'SENSITIVE',
        
        # 5. SYSTEM/TECHNICAL PLACEHOLDERS[8] (from IT systems)
        'system error', 'System Error', 'SYSTEM ERROR',
        'connection error', 'Connection Error', 'CONNECTION ERROR',
        'timeout', 'Timeout', 'TIMEOUT', 'expired', 'Expired', 'EXPIRED',
        'unavailable service', 'Unavailable Service', 'UNAVAILABLE SERVICE',
        'service down', 'Service Down', 'SERVICE DOWN',
        'maintenance', 'Maintenance', 'MAINTENANCE',
        'auto-generated', 'Auto-Generated', 'AUTO-GENERATED',
        'system-assigned', 'System-Assigned', 'SYSTEM-ASSIGNED',
        'auto-filled', 'Auto-Filled', 'AUTO-FILLED',
        'calculated', 'Calculated', 'CALCULATED',
        'derived', 'Derived', 'DERIVED', 'computed', 'Computed', 'COMPUTED',
        
        # 6. IMPORT/EXPORT ARTIFACTS[6] (from data migration issues)
        'encoding error', 'Encoding Error', 'ENCODING ERROR',
        'conversion failed', 'Conversion Failed', 'CONVERSION FAILED',
        'import error', 'Import Error', 'IMPORT ERROR',
        'export error', 'Export Error', 'EXPORT ERROR',
        'parsing error', 'Parsing Error', 'PARSING ERROR',
        'format error', 'Format Error', 'FORMAT ERROR',
        'corrupted', 'Corrupted', 'CORRUPTED',
        'truncated', 'Truncated', 'TRUNCATED',
        'malformed', 'Malformed', 'MALFORMED',
        'invalid format', 'Invalid Format', 'INVALID FORMAT',
        
        # 7. LEGACY SYSTEM CODES[15] (from old databases)
        'legacy code', 'Legacy Code', 'LEGACY CODE',
        'deprecated', 'Deprecated', 'DEPRECATED',
        'obsolete', 'Obsolete', 'OBSOLETE',
        'retired', 'Retired', 'RETIRED', 'discontinued', 'Discontinued', 'DISCONTINUED',
        'end of life', 'End Of Life', 'END OF LIFE',
        'no longer valid', 'No Longer Valid', 'NO LONGER VALID',
        'superseded', 'Superseded', 'SUPERSEDED',
        'replaced', 'Replaced', 'REPLACED',
        
        # 8. GEOGRAPHIC/DEMOGRAPHIC SPECIFIC
        'not disclosed', 'Not Disclosed', 'NOT DISCLOSED',
        'location withheld', 'Location Withheld', 'LOCATION WITHHELD',
        'address confidential', 'Address Confidential', 'ADDRESS CONFIDENTIAL',
        'nomadic', 'Nomadic', 'NOMADIC', 'transient', 'Transient', 'TRANSIENT',
        'homeless', 'Homeless', 'HOMELESS', 'no fixed address', 'No Fixed Address',
        'international', 'International', 'INTERNATIONAL',
        'global', 'Global', 'GLOBAL', 'worldwide', 'Worldwide', 'WORLDWIDE',
        
        # 9. EXTENDED NUMERIC PLACEHOLDERS[15] (common in databases)
        '-1', '-99', '-999', '-9999', '-99999', 
        '99', '999', '9999', '99999', '999999',
        '0', '00', '000', '0000', '00000',
        '8888', '88888', '7777', '77777',
        '1111', '11111', '2222', '22222',
        '9999-12-31', '1900-01-01', '1111-11-11', '0000-00-00',
        
        # 10. SPECIAL CHARACTERS AND SYMBOLS[8] (extended)
        '-', '--', '---', '----', '-----',
        '?', '??', '???', '????', '?????',
        '.', '..', '...', '....', '.....',
        '*', '**', '***', '****', '*****',
        '#', '##', '###', '####', '#####',
        '_', '__', '___', '____', '_____',
        '|', '||', '|||', '||||', '|||||',
        '/', '//', '///', '////', '/////',
        '\\', '\\\\', '\\\\\\', '\\\\\\\\', '\\\\\\\\\\',
        'xxx', 'xxxx', 'xxxxx', 'xxxxxx',
        'XXX', 'XXXX', 'XXXXX', 'XXXXXX',
        'zzz', 'zzzz', 'zzzzz', 'zzzzzz',
        'ZZZ', 'ZZZZ', 'ZZZZZ', 'ZZZZZZ',
        
        # 11. EXCEL/SPREADSHEET SPECIFIC ERRORS (extended)
        '#n/a', '#N/A', '#value!', '#VALUE!', '#error!', '#ERROR!',
        '#ref!', '#REF!', '#name?', '#NAME?', '#div/0!', '#DIV/0!',
        '#num!', '#NUM!', '#null!', '#NULL!', '#getting_data', '#GETTING_DATA',
        '#spill!', '#SPILL!', '#calc!', '#CALC!', '#busy!', '#BUSY!',
        '#blocked!', '#BLOCKED!', '#connect!', '#CONNECT!',
        
        # 12. INTERNATIONAL/MULTILINGUAL[6] (expanded)
        # Spanish
        'desconocido', 'Desconocido', 'DESCONOCIDO',
        'no disponible', 'No Disponible', 'NO DISPONIBLE',
        'sin datos', 'Sin Datos', 'SIN DATOS',
        'sin información', 'Sin Información', 'SIN INFORMACIÓN',
        'otros', 'Otros', 'OTROS', 'otra', 'Otra', 'OTRA',
        'varios', 'Varios', 'VARIOS', 'múltiple', 'Múltiple', 'MÚLTIPLE',
        
        # French
        'inconnu', 'Inconnu', 'INCONNU',
        'pas disponible', 'Pas Disponible', 'PAS DISPONIBLE',
        'non disponible', 'Non Disponible', 'NON DISPONIBLE',
        'autres', 'Autres', 'AUTRES', 'autre', 'Autre', 'AUTRE',
        'divers', 'Divers', 'DIVERS', 'varié', 'Varié', 'VARIÉ',
        
        # German
        'unbekannt', 'Unbekannt', 'UNBEKANNT',
        'nicht verfügbar', 'Nicht Verfügbar', 'NICHT VERFÜGBAR',
        'nicht angegeben', 'Nicht Angegeben', 'NICHT ANGEGEBEN',
        'andere', 'Andere', 'ANDERE', 'sonstige', 'Sonstige', 'SONSTIGE',
        'verschiedene', 'Verschiedene', 'VERSCHIEDENE',
        
        # Portuguese
        'não disponível', 'Não Disponível', 'NÃO DISPONÍVEL',
        'não informado', 'Não Informado', 'NÃO INFORMADO',
        'desconhecido', 'Desconhecido', 'DESCONHECIDO',
        'outros', 'Outros', 'OUTROS', 'outras', 'Outras', 'OUTRAS',
        'vários', 'Vários', 'VÁRIOS', 'múltiplos', 'Múltiplos', 'MÚLTIPLOS',
        
        # Italian
        'sconosciuto', 'Sconosciuto', 'SCONOSCIUTO',
        'non disponibile', 'Non Disponibile', 'NON DISPONIBILE',
        'non specificato', 'Non Specificato', 'NON SPECIFICATO',
        'altri', 'Altri', 'ALTRI', 'altro', 'Altro', 'ALTRO',
        'vari', 'Vari', 'VARI', 'misto', 'Misto', 'MISTO',
        
        # 13. COMMON TYPOS AND VARIATIONS (research-based)
        'otehrs', 'ohters', 'toher', 'otehr', 'htoer',
        'unknwon', 'unkwnon', 'nknown', 'uknwon', 'unknonw',
        'misssing', 'mising', 'missng', 'missin', 'mssing',
        'emtpy', 'empyt', 'empy', 'eptmy', 'epmty',
        'nul', 'nill', 'nul', 'nulle', 'nulL',
        'blnak', 'balnk', 'blanck', 'blannk', 'blankk',
        'genereal', 'genreal', 'genaral', 'generall', 'genearl',
        'varous', 'varius', 'varuous', 'vairious', 'variuos',
        'miscelaneous', 'miscellanous', 'miscellenaous', 'miscelanous',
        
        # 14. CONTEXT-SPECIFIC TERMS (domain-specific)
        # HR/Employment
        'position eliminated', 'Position Eliminated', 'POSITION ELIMINATED',
        'role restructured', 'Role Restructured', 'ROLE RESTRUCTURED',
        'department dissolved', 'Department Dissolved', 'DEPARTMENT DISSOLVED',
        'contractor', 'Contractor', 'CONTRACTOR',
        'consultant', 'Consultant', 'CONSULTANT',
        'freelancer', 'Freelancer', 'FREELANCER',
        
        # Financial
        'amount confidential', 'Amount Confidential', 'AMOUNT CONFIDENTIAL',
        'price on request', 'Price On Request', 'PRICE ON REQUEST',
        'quote required', 'Quote Required', 'QUOTE REQUIRED',
        'negotiable', 'Negotiable', 'NEGOTIABLE',
        'market rate', 'Market Rate', 'MARKET RATE',
        'competitive', 'Competitive', 'COMPETITIVE',
        
        # Educational/Academic
        'course discontinued', 'Course Discontinued', 'COURSE DISCONTINUED',
        'program suspended', 'Program Suspended', 'PROGRAM SUSPENDED',
        'grade pending', 'Grade Pending', 'GRADE PENDING',
        'credit transfer', 'Credit Transfer', 'CREDIT TRANSFER',
        'audit', 'Audit', 'AUDIT', 'pass/fail', 'Pass/Fail', 'PASS/FAIL',
        
        # Medical/Healthcare  
        'patient declined', 'Patient Declined', 'PATIENT DECLINED',
        'contraindicated', 'Contraindicated', 'CONTRAINDICATED',
        'not tested', 'Not Tested', 'NOT TESTED',
        'result pending', 'Result Pending', 'RESULT PENDING',
        'lab error', 'Lab Error', 'LAB ERROR',
        'specimen insufficient', 'Specimen Insufficient', 'SPECIMEN INSUFFICIENT',
        
        # 15. ENCODING AND CHARACTER ISSUES
        '�', '��', '���', 'Ã¢â‚¬â„¢', 'Ã¢â‚¬Å"',
        'â€™', 'â€œ', 'â€', 'â€¦', 'â€"',
        'Ã©', 'Ã¡', 'Ã­', 'Ã³', 'Ãº',
        'Ã±', 'Ã§', 'Ã¼', 'Ã¶', 'Ã¤',
        
        # ALL PREVIOUSLY COVERED TERMS (from the conversation history)
        # ... [keeping all the extensive terms from before]
    ]
    
    # Remove duplicates while preserving order and strip whitespace
    seen = set()
    unique_terms = []
    for term in invalid_terms:
        if term and term.strip() and term.strip() not in seen:
            seen.add(term.strip())
            unique_terms.append(term.strip())
    
    return unique_terms

# Usage in your validation function:
def create_invalid_terms_list():
    """Enhanced version with comprehensive invalid terms"""
    return create_comprehensive_invalid_terms_list()


def check_invalid_data(df, columns_to_check, custom_invalid_terms=None):
    """
    Check specified columns for blank data and invalid placeholder values
    
    Parameters:
    - df: DataFrame to check
    - columns_to_check: List of column names to validate
    - custom_invalid_terms: Additional invalid terms to check for
    
    Returns:
    - Dictionary with validation results
    """
    
    # Get invalid terms list
    invalid_terms = create_invalid_terms_list()
    if custom_invalid_terms:
        invalid_terms.extend([term.lower().strip() for term in custom_invalid_terms])
    
    # Remove duplicates
    invalid_terms = list(set(invalid_terms))
    
    print("Checking for invalid data...")
    print(f"Invalid terms being checked: {len(invalid_terms)} terms")
    print(f"Columns to check: {columns_to_check}")
    
    validation_results = {}
    
    for col in columns_to_check:
        if col not in df.columns:
            print(f"⚠️  Warning: Column '{col}' not found in dataset")
            continue
        
        col_results = {
            'column_name': col,
            'total_rows': len(df),
            'blank_count': 0,
            'invalid_count': 0,
            'blank_indices': [],
            'invalid_indices': [],
            'invalid_values_found': [],
            'clean_count': 0
        }
        
        # Check for blank/null values
        blank_mask = df[col].isnull() | (df[col].astype(str).str.strip() == '')
        col_results['blank_count'] = blank_mask.sum()
        col_results['blank_indices'] = df[blank_mask].index.tolist()
        
        # Check for invalid terms
        invalid_indices = []
        invalid_values = []
        
        for idx, value in df[col].items():
            if pd.isna(value) or str(value).strip() == '':
                continue  # Already counted as blank
                
            value_str = str(value).lower().strip()
            
            # Check exact matches first
            if value_str in invalid_terms:
                invalid_indices.append(idx)
                invalid_values.append(str(value))
            else:
                # Check partial matches for longer invalid terms
                for term in invalid_terms:
                    if len(term) > 2 and term in value_str:
                        invalid_indices.append(idx)
                        invalid_values.append(str(value))
                        break
        
        col_results['invalid_count'] = len(invalid_indices)
        col_results['invalid_indices'] = invalid_indices
        col_results['invalid_values_found'] = list(set(invalid_values))
        col_results['clean_count'] = len(df) - col_results['blank_count'] - col_results['invalid_count']
        
        validation_results[col] = col_results
        
        # Print column summary
        print(f"\n📊 Column '{col}' Results:")
        print(f"   ├─ Total rows: {col_results['total_rows']:,}")
        print(f"   ├─ Blank/Empty: {col_results['blank_count']:,} ({col_results['blank_count']/len(df)*100:.1f}%)")
        print(f"   ├─ Invalid data: {col_results['invalid_count']:,} ({col_results['invalid_count']/len(df)*100:.1f}%)")
        print(f"   └─ Clean data: {col_results['clean_count']:,} ({col_results['clean_count']/len(df)*100:.1f}%)")
        
        if col_results['invalid_values_found']:
            print(f"   📝 Invalid values found: {col_results['invalid_values_found'][:10]}{'...' if len(col_results['invalid_values_found']) > 10 else ''}")
    
    return validation_results

def separate_data_by_validity(df, columns_to_check, custom_invalid_terms=None, output_file="Data_Separated.xlsx"):
    """
    Separate data into clean and invalid data sheets based on validation results
    """
    
    print("\n" + "="*60)
    print("SEPARATING DATA BY VALIDITY")
    print("="*60)
    
    # Get validation results
    validation_results = check_invalid_data(df, columns_to_check, custom_invalid_terms)
    
    # Collect all problematic indices
    all_problematic_indices = set()
    
    for col, results in validation_results.items():
        if col not in df.columns:
            continue
        all_problematic_indices.update(results['blank_indices'])
        all_problematic_indices.update(results['invalid_indices'])
    
    # Separate data
    problematic_rows = df.loc[list(all_problematic_indices)].copy()
    clean_rows = df.drop(all_problematic_indices).reset_index(drop=True)
    
    print(f"\n📋 DATA SEPARATION RESULTS:")
    print(f"   ├─ Total original rows: {len(df):,}")
    print(f"   ├─ Clean rows: {len(clean_rows):,} ({len(clean_rows)/len(df)*100:.1f}%)")
    print(f"   └─ Problematic rows: {len(problematic_rows):,} ({len(problematic_rows)/len(df)*100:.1f}%)")
    
    # Create detailed problem analysis for problematic rows
    problem_analysis = []
    for idx in all_problematic_indices:
        row_problems = []
        for col, results in validation_results.items():
            if col not in df.columns:
                continue
            if idx in results['blank_indices']:
                row_problems.append(f"{col}: BLANK")
            elif idx in results['invalid_indices']:
                value = str(df.loc[idx, col])
                row_problems.append(f"{col}: INVALID ({value})")
        
        problem_analysis.append({
            'Original_Index': idx,
            'Problems_Found': '; '.join(row_problems),
            'Problem_Count': len(row_problems)
        })
    
    problem_df = pd.DataFrame(problem_analysis)
    
    # Save to Excel with multiple sheets
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        # Clean data sheet
        clean_rows.to_excel(writer, sheet_name='Clean_Data', index=False)
        
        # Problematic data sheet
        problematic_rows.to_excel(writer, sheet_name='Problematic_Data', index=False)
        
        # Problem analysis sheet
        problem_df.to_excel(writer, sheet_name='Problem_Analysis', index=False)
        
        # Validation summary sheet
        summary_data = []
        for col, results in validation_results.items():
            if col in df.columns:
                summary_data.append({
                    'Column': col,
                    'Total_Rows': results['total_rows'],
                    'Blank_Count': results['blank_count'],
                    'Invalid_Count': results['invalid_count'],
                    'Clean_Count': results['clean_count'],
                    'Blank_Percentage': round(results['blank_count']/results['total_rows']*100, 2),
                    'Invalid_Percentage': round(results['invalid_count']/results['total_rows']*100, 2),
                    'Invalid_Values_Sample': str(results['invalid_values_found'][:5])
                })
        
        pd.DataFrame(summary_data).to_excel(writer, sheet_name='Validation_Summary', index=False)
    
    print(f"\n✅ Data separated and saved to: {output_file}")
    print(f"\n📚 Excel file contains 4 sheets:")
    print(f"   • Clean_Data: {len(clean_rows):,} clean rows")
    print(f"   • Problematic_Data: {len(problematic_rows):,} rows with issues")
    print(f"   • Problem_Analysis: Detailed problem breakdown per row")
    print(f"   • Validation_Summary: Column-wise validation statistics")
    
    return clean_rows, problematic_rows, validation_results

def validate_and_separate_excel(file_path, columns_to_check, custom_invalid_terms=None, output_file=None):
    """
    Main function to load Excel file, validate specified columns, and separate data
    """
    
    print("="*80)
    print("DATA VALIDATION AND SEPARATION TOOL")
    print("="*80)
    
    # Load data
    print(f"\n📁 Loading data from: {file_path}")
    try:
        df = pd.read_excel(file_path)
        print(f"✅ Successfully loaded {len(df):,} rows and {len(df.columns)} columns")
    except Exception as e:
        print(f"❌ Error loading file: {e}")
        return None, None, None
    
    # Validate column names
    missing_cols = [col for col in columns_to_check if col not in df.columns]
    if missing_cols:
        print(f"\n⚠️  Warning: These columns were not found: {missing_cols}")
        print(f"Available columns: {list(df.columns)}")
        columns_to_check = [col for col in columns_to_check if col in df.columns]
    
    if not columns_to_check:
        print("❌ No valid columns to check!")
        return None, None, None
    
    # Set output file name
    if output_file is None:
        base_name = file_path.rsplit('.', 1)[0]
        output_file = f"{base_name}_Validated.xlsx"
    
    # Perform validation and separation
    clean_data, problematic_data, validation_results = separate_data_by_validity(
        df, columns_to_check, custom_invalid_terms, output_file
    )
    
    return clean_data, problematic_data, validation_results

# Usage Examples:

# Example 1: Basic usage with common columns
# columns_to_validate = ['Job_Function', 'Department', 'Location', 'Company']
# clean_df, problem_df, results = validate_and_separate_excel(
#     "GISMS July'25.xlsx", 
#     columns_to_validate
# )

# # Example 2: With custom invalid terms
# custom_terms = ['to be updated', 'pending approval', 'temporary assignment']
# clean_df, problem_df, results = validate_and_separate_excel(
#     "GISMS July'25.xlsx", 
#     ['Job_Function', 'Department'], 
#     custom_invalid_terms=custom_terms,
#     output_file="GISMS_Clean_Data.xlsx"
# )

# Example 3: Check single column thoroughly
# single_column_results = validate_and_separate_excel(
#     "GISMS July'25.xlsx", 
#     ['Job_Function']
# )


In [40]:
# For quick column validation without separation shows what invalid data is there without excel file
def quick_validate_columns_with_values(df, columns):
    """Quick validation report for specified columns showing actual invalid values"""
    
    invalid_terms = create_invalid_terms_list()
    
    for col in columns:
        if col not in df.columns:
            print(f"Column '{col}' not found!")
            continue
        
        # Count blank/empty values
        blank_count = df[col].isnull().sum() + (df[col].astype(str).str.strip() == '').sum()
        
        # Collect invalid values with their counts
        invalid_values_counter = {}
        
        for value in df[col].dropna():
            val_str = str(value).lower().strip()
            if val_str in invalid_terms:
                # Store original value (not lowercase) to show actual data
                invalid_values_counter[value] = invalid_values_counter.get(value, 0) + 1
        
        invalid_count = sum(invalid_values_counter.values())
        clean_count = len(df) - blank_count - invalid_count
        
        print(f"\n📊 {col}:")
        print(f"  Blank: {blank_count} | Invalid: {invalid_count} | Clean: {clean_count}")
        
        # Show actual invalid values found
        if invalid_values_counter:
            print(f"  ⚠️  Invalid values found:")
            for invalid_value, count in sorted(invalid_values_counter.items(), key=lambda x: x[1], reverse=True):
                print(f"    • '{invalid_value}': {count} times")
        else:
            print(f"  ✅ No invalid placeholder values found")

# Usage with your columns:
quick_validate_columns_with_values(df, [
    'Job_Function', 
    'Cleaned Job_Seniority',
    'Designation',
    'Job Profile',
    'Employees_Range',
    'Turnover_Range',
    'Industry',
    "Parent_Industry"
    ,"Type"
])




📊 Job_Function:
  Blank: 1 | Invalid: 7994 | Clean: 25790
  ⚠️  Invalid values found:
    • 'Others': 7994 times

📊 Cleaned Job_Seniority:
  Blank: 0 | Invalid: 0 | Clean: 33785
  ✅ No invalid placeholder values found

📊 Designation:
  Blank: 0 | Invalid: 8 | Clean: 33777
  ⚠️  Invalid values found:
    • 'Nil': 3 times
    • 'Research': 2 times
    • 'Xxx': 2 times
    • 'Confidential': 1 times

📊 Job Profile:
  Blank: 0 | Invalid: 273 | Clean: 33512
  ⚠️  Invalid values found:
    • 'Not Available': 273 times

📊 Employees_Range:
  Blank: 0 | Invalid: 0 | Clean: 33785
  ✅ No invalid placeholder values found

📊 Turnover_Range:
  Blank: 0 | Invalid: 0 | Clean: 33785
  ✅ No invalid placeholder values found

📊 Industry:
  Blank: 0 | Invalid: 1942 | Clean: 31843
  ⚠️  Invalid values found:
    • 'Miscellaneous': 1853 times
    • 'Various': 70 times
    • 'Research': 19 times

📊 Parent_Industry:
  Blank: 0 | Invalid: 1923 | Clean: 31862
  ⚠️  Invalid values found:
    • 'Miscellaneous': 19

In [ ]:
# Specify the exact columns you want to check
columns_to_check = ['Job_Function', 'Department', 'Company']  # Add your column names

# Run the validation and separation
clean_data, problematic_data, validation_results = validate_and_separate_excel(
    "GISMS July'25.xlsx",
    columns_to_check,
    output_file="GISMS_Cleaned.xlsx"
)


In [47]:
def get_unique_values_simple(df, columns):
    """
    Simple function to return unique values for specified columns
    """
    unique_results = {}
    
    for col in columns:
        if col in df.columns:
            unique_values = df[col].unique()
            # Remove NaN values and convert to list
            unique_clean = [val for val in unique_values if pd.notna(val)]
            unique_results[col] = unique_clean
        else:
            print(f"Column '{col}' not found!")
    
    return unique_results

# Usage:
columns_to_check = ['Job_Function', 'Parent_Industry','Cleaned Job_Seniority']
unique_vals = get_unique_values_simple(df, columns_to_check)

# Print results
for col, values in unique_vals.items():
    print(f"\n{col}: {len(values)} unique values")
    print(f"Values: {values}")



Job_Function: 49 unique values
Values: ['Operations', 'Professor', 'IT Specialist', 'IT', 'CXO', 'Marketing', 'CXO-1 (Head/GM/AGM/DGM)', 'CXO (CEO,MD,Chairman,Director,Owner,C Level,Proprietor,VP)', 'IT/Cloud/Architect/Technology-Executive', 'Business', 'Sales', 'Developer', 'HR', 'Risk', 'Finance', 'Project Management', 'Procurement', 'Manager- Senior Manager/Manager/DM/AM and etc', 'Design', 'Account', 'Product', 'Eng/Designer/Developer/Exec/Administrator', 'Designer', 'Data', 'Sales & Marketing', 'Scientist', 'Digital', 'Engineering', 'Secretary', 'Delivery', 'Program Management', 'Logistics', 'Revenue', 'R&D', 'Supply Chain', 'Strategy', 'E-Commerce', 'Infrastructure', 'Brand', 'Commissioner', 'Plant', 'Security', 'Production', 'Customer Service', 'Analytics', 'Creative', 'Scientific', 'People', 'Representative']

Parent_Industry: 24 unique values
Values: ['Healthcare', 'Education', 'Engineering', 'Information & Technology', 'Travel & Transportation', 'Professional Services', 'Con

In [56]:


def create_simple_unique_values_excel_with_counts(df, columns, output_file="Simple_Unique_Values_With_Counts.xlsx"):
    """
    Create Excel file with:
    - First sheet: column names and their unique values
    - Second sheet: count of each unique value for each column
    """
    
    # Get unique values for each column (First Sheet)
    unique_data = {}
    max_length = 0
    
    for col in columns:
        if col in df.columns:
            unique_values = df[col].dropna().unique().tolist()
            unique_data[col] = unique_values
            max_length = max(max_length, len(unique_values))
        else:
            print(f"Column '{col}' not found!")
    
    # Prepare data for first sheet (unique values)
    excel_data = {}
    for col, values in unique_data.items():
        padded_values = values + [''] * (max_length - len(values))
        excel_data[col] = padded_values
    
    # Prepare data for second sheet (counts)
    counts_data = {}
    max_count_length = 0
    
    for col in columns:
        if col in df.columns:
            value_counts = df[col].value_counts(dropna=False)
            
            # Create columns for value and count
            counts_data[f"{col}_Value"] = list(value_counts.index)
            counts_data[f"{col}_Count"] = list(value_counts.values)
            
            max_count_length = max(max_count_length, len(value_counts))
    
    # Pad counts data to equal length
    for key in counts_data:
        counts_data[key] += [''] * (max_count_length - len(counts_data[key]))
    
    # Create DataFrames
    unique_values_df = pd.DataFrame(excel_data)
    counts_df = pd.DataFrame(counts_data)
    
    # Write to Excel with two sheets
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        unique_values_df.to_excel(writer, index=False, sheet_name='Unique_Values')
        counts_df.to_excel(writer, index=False, sheet_name='Value_Counts')
    
    print(f"✅ Excel file created: {output_file}")
    print(f"📊 Sheet 1 - Unique Values: {len(columns)} columns")
    print(f"📊 Sheet 2 - Value Counts: Value and count pairs for each column")
    
    for col in unique_data:
        print(f"   • {col}: {len(unique_data[col])} unique values")
    
    return unique_values_df, counts_df

# Usage with your columns
columns_to_check = ['Job_Function', 'Parent_Industry', 'Cleaned Job_Seniority']
# Create Excel with both sheets
unique_df, counts_df = create_simple_unique_values_excel_with_counts(df, columns_to_check, "Job_Security_Unique_Values_With_Counts.xlsx")

# Preview what will be in Excel
print("\n📋 PREVIEW OF SHEET 1 - UNIQUE VALUES:")
print(unique_df.head(10))

print("\n📋 PREVIEW OF SHEET 2 - VALUE COUNTS:")
print(counts_df.head(10))


✅ Excel file created: Job_Security_Unique_Values_With_Counts.xlsx
📊 Sheet 1 - Unique Values: 3 columns
📊 Sheet 2 - Value Counts: Value and count pairs for each column
   • Job_Function: 49 unique values
   • Parent_Industry: 24 unique values
   • Cleaned Job_Seniority: 6 unique values

📋 PREVIEW OF SHEET 1 - UNIQUE VALUES:
                                        Job_Function  \
0                                         Operations   
1                                          Professor   
2                                      IT Specialist   
3                                                 IT   
4                                                CXO   
5                                          Marketing   
6                            CXO-1 (Head/GM/AGM/DGM)   
7  CXO (CEO,MD,Chairman,Director,Owner,C Level,Pr...   
8            IT/Cloud/Architect/Technology-Executive   
9                                           Business   

            Parent_Industry Cleaned Job_Seniority  
0     